# Qwen2.5 DepEd Grade 11-12 Math LoRA Fine-tuning (Colab)
Manual checkpoints: Colab login/runtime, GPU selection, dataset file presence, HF token entry if needed.

## 1. Set Up Python Environment and Imports (Cell A)

In [ ]:
%pip -q install --upgrade pip
%pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip -q install trl==0.22.2 transformers==4.57.3 accelerate peft datasets huggingface_hub bitsandbytes

import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

import unsloth
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from huggingface_hub import HfApi, create_repo, login
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Create Project Configuration Variables (Cell B)

In [ ]:
HF_TOKEN = ''  # TODO
HF_USERNAME = 'Deign86'  # TODO
DATA_PATH = 'Deign86/deped-math-sft-dataset'  # HF dataset repo id or local jsonl path
OUTPUT_DIR = '/content/qwen2.5-deped-math-lora'
REPO_ID = f'{HF_USERNAME}/qwen2.5-deped-math-instruct-lora'
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
MAX_SEQ_LENGTH = 2048
NUM_TRAIN_EPOCHS = 2
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
SEED = 3407
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
CREATE_DATASET_REPO = False
DATASET_REPO_ID = f'{HF_USERNAME}/deped-math-sft-dataset'


## 3. Define Core Data Structures

In [ ]:
@dataclass
class TrainingConfig:
    hf_token: str
    hf_username: str
    data_path: str
    output_dir: str
    repo_id: str
    model_name: str
    max_seq_length: int
    num_train_epochs: int

REQUIRED_LEGACY_COLUMNS = {'system', 'user', 'assistant'}
cfg = TrainingConfig(HF_TOKEN, HF_USERNAME, DATA_PATH, OUTPUT_DIR, REPO_ID, MODEL_NAME, MAX_SEQ_LENGTH, NUM_TRAIN_EPOCHS)


## 4. Implement Initial Business Logic Functions (Cells C-D)

In [ ]:
def setup_hf_and_repos(cfg: TrainingConfig) -> None:
    if cfg.hf_token.strip():
        login(token=cfg.hf_token, add_to_git_credential=False)
    else:
        print('HF_TOKEN empty. Use notebook_login if required.')
    create_repo(repo_id=cfg.repo_id, repo_type='model', exist_ok=True)

def to_qwen_chatml(system: str, user: str, assistant: str) -> str:
    return ('<|im_start|>system\n' + system.strip() + '\n<|im_end|>\n' + '<|im_start|>user\n' + user.strip() + '\n<|im_end|>\n' + '<|im_start|>assistant\n' + assistant.strip() + '\n<|im_end|>')

def messages_to_chatml(messages: List[Dict[str, Any]]) -> str:
    chunks: List[str] = []
    for msg in messages:
        role = str(msg.get('role', '')).strip()
        content = str(msg.get('content', '')).strip()
        if role in {'system', 'user', 'assistant'} and content:
            chunks.append(f'<|im_start|>{role}\n{content}\n<|im_end|>')
    return '\n'.join(chunks)

def load_and_format_dataset(data_path: str):
    if Path(data_path).exists():
        ds = load_dataset('json', data_files=data_path, split='train')
    elif '/' in data_path and not data_path.startswith('/'):
        ds = load_dataset(data_path, split='train')
    else:
        raise FileNotFoundError(f'Missing dataset: {data_path}. Use a local JSONL path or a Hub dataset repo id like user/name.')
    cols = set(ds.column_names)
    def _map_fn(ex: Dict[str, Any]) -> Dict[str, str]:
        if isinstance(ex.get('text'), str) and ex['text'].strip():
            return {'text': ex['text']}
        if isinstance(ex.get('chatml'), str) and ex['chatml'].strip():
            return {'text': ex['chatml']}
        if REQUIRED_LEGACY_COLUMNS.issubset(cols):
            return {'text': to_qwen_chatml(ex['system'], ex['user'], ex['assistant'])}
        if isinstance(ex.get('messages'), list):
            chatml = messages_to_chatml(ex['messages'])
            if chatml.strip():
                return {'text': chatml}
        raise ValueError('Row missing supported SFT fields: expected text/chatml/messages or system+user+assistant.')
    out = ds.map(_map_fn, remove_columns=ds.column_names)
    print('Rows:', len(out))
    print('Sample:\n', out[0]['text'][:1200])
    return out


## 5. Wire Functions into an Executable Workflow (Cells C-F)

In [ ]:
setup_hf_and_repos(cfg)
train_dataset = load_and_format_dataset(cfg.data_path)
model, tokenizer = FastLanguageModel.from_pretrained(model_name=cfg.model_name, max_seq_length=cfg.max_seq_length, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=LORA_R, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none', use_gradient_checkpointing='unsloth', random_state=SEED)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
trainer_args = SFTConfig(output_dir=cfg.output_dir, dataset_text_field='text', max_length=cfg.max_seq_length, per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, warmup_steps=5, num_train_epochs=cfg.num_train_epochs, learning_rate=LEARNING_RATE, logging_steps=1, optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='linear', seed=SEED, fp16=not use_bf16, bf16=use_bf16, report_to='none', save_strategy='steps', save_steps=50, save_total_limit=2)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_dataset, args=trainer_args)
train_result = trainer.train()
print(train_result.metrics)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)


## 6. Add Basic Logging and Error Handling (Cell G)

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('push')
fallback_cmds = f'''# Fallback commands
hf auth login
hf upload {cfg.repo_id} {cfg.output_dir} . --repo-type model --commit-message "Upload LoRA adapter"
'''
try:
    if cfg.hf_token.strip():
        login(token=cfg.hf_token, add_to_git_credential=False)
    model.push_to_hub(cfg.repo_id)
    tokenizer.push_to_hub(cfg.repo_id)
except Exception as e:
    logger.exception('Push failed: %s', repr(e))
    print(fallback_cmds)


## 7. Write Unit Tests for Core Functions

In [ ]:
import unittest

class TestCoreFormatting(unittest.TestCase):
    def test_chatml_has_roles(self):
        txt = to_qwen_chatml('sys', 'usr', 'asst')
        self.assertIn('<|im_start|>system', txt)
        self.assertIn('<|im_start|>user', txt)
        self.assertIn('<|im_start|>assistant', txt)

def run_unit_tests() -> None:
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCoreFormatting)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError('Core unit tests failed')


## 8. Run Tests and Inspect Output (Cell H)

In [ ]:
run_unit_tests()
base_model, infer_tokenizer = FastLanguageModel.from_pretrained(model_name=cfg.model_name, max_seq_length=cfg.max_seq_length, dtype=None, load_in_4bit=True)
infer_model = PeftModel.from_pretrained(base_model, cfg.output_dir)
FastLanguageModel.for_inference(infer_model)
messages = [
    {'role':'system','content':'You are a precise DepEd SHS General Math tutor. Use numbered steps, show derivations, include a short self-check, then final answer.'},
    {'role':'user','content':'A rectangle has perimeter 54 cm and length 3 cm more than width. Find length and width.'}
]
prompt = infer_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = infer_tokenizer([prompt], return_tensors='pt').to(infer_model.device)
with torch.no_grad():
    outputs = infer_model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=infer_tokenizer.eos_token_id, pad_token_id=infer_tokenizer.eos_token_id)
completion = infer_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(completion)
